# Assignment 1: Sampling and Reproducibility

The code at the end of this file explores contact tracing data about an outbreak of the flu, and demonstrates the dangers of incomplete and non-random samples. This assignment is modified from [Contact tracing can give a biased sample of COVID-19 cases](https://andrewwhitby.com/2020/11/24/contact-tracing-biased/) by Andrew Whitby.

Examine the code below. Identify all stages at which sampling is occurring in the model. Describe in words the sampling procedure, referencing the functions used, sample size, sampling frame, any underlying distributions involved. 


Stage A: “Sampling” the population / sampling frame

Sampling frame: all attendees in ppl, size N = 1000
referenced code:

events = ['wedding'] * 200 + ['brunch'] * 800
ppl = pd.DataFrame(...)


This isn’t random sampling yet; it defines the population composition: 20% wedding, 80% brunch.

Stage B — Infection assignment (random sample without replacement)

What is sampled? Which individuals become infected.
Function used: np.random.choice(...)
Sample size: int(len(ppl) * ATTACK_RATE) = int(1000 * 0.10) = 100
Sampling frame: ppl.index (0…999)
Replacement: replace=False → no person can be infected twice
Underlying distribution: equivalent to drawing a simple random sample without replacement of size 100 (so the number infected is fixed at 100 each run)

referenced code:
infected_indices = np.random.choice(ppl.index, size=100, replace=False)

Stage C — Primary contact tracing among infected (Bernoulli trials)

What is sampled? Which infected individuals get traced in primary tracing.
Function used: np.random.rand(k) < TRACE_SUCCESS
Sample size: k = sum(ppl['infected']) = 100
Sampling frame: infected individuals only
Underlying distribution: for each infected person, traced status is a Bernoulli(p=0.20) draw, independent across infected people.

referenced code:
ppl.loc[ppl['infected'], 'traced'] = np.random.rand(100) < 0.20

Stage D — Secondary tracing trigger (threshold rule based on a biased observed sample)

What is sampled? Not a new RNG draw, but a filtering rule based on the random outcome of Stage C.
Function used: value_counts() then thresholding:
event_trace_counts = ppl[ppl['traced'] == True]['event'].value_counts()
events_traced = event_trace_counts[event_trace_counts >= 2].index

Sampling frame: the set of traced infected people (a non-random subset of infected, produced by Stage C).
Mechanism: if an event type has ≥2 traced cases, then all infected people at that event type become traced:
ppl.loc[ppl['event'].isin(events_traced) & ppl['infected'], 'traced'] = True

Stage E — Repetition sampling (Monte Carlo)
What is sampled? You repeat the whole stochastic process R times:
results = [simulate_event(m) for m in range(1000)]




Modify the number of repetitions in the simulation to 10 and 100 (from the original 1000). Run the script multiple times and observe the outputted graphs. Comment on the reproducibility of the results.

results = [simulate_event(m) for m in range(10)] # Run multiple times
results = [simulate_event(m) for m in range(10)] # Run multiple times

With 10 repetitions: the histograms look very jumpy run-to-run. Bars appear/disappear, and the “center” of the distribution can shift noticeably. This is because we have a small Monte Carlo sample, so the estimate of the underlying distribution has high variance. With 100 repetitions: results are somewhat more stable but still change slightly between runs. The overall shape becomes clearer, but the peak location and tails still vary.

Alter the code so that it is reproducible. Describe the changes you made to the code and how they affected the reproducibility of the script. The script needs to produce the same output when run multiple times.

The best and easiest practice is: use a dedicated RNG and pass it into the simulation, instead of relying on the global np.random state.
Step 1 — add a seed constant near the top
SEED = 42
Step 2 — update simulate_event to accept an RNG
def simulate_event(m, rng):
Step 3 — replace random calls
infected_indices = rng.choice(ppl.index, size=int(len(ppl) * ATTACK_RATE), replace=False)
ppl.loc[ppl['infected'], 'traced'] = rng.random(sum(ppl['infected'])) < TRACE_SUCCESS
Step 4 — create the RNG once and use it
rng = np.random.default_rng(SEED)
results = [simulate_event(m, rng) for m in range(1000)]

Introduced a fixed seed (SEED = 42) and used a local random number generator (np.random.default_rng(SEED)). All random draws now come from rng, so every run generates the same sequence of random numbers, producing identical infection assignments, tracing outcomes, and therefore identical histograms. As a result, the script becomes fully reproducible: running the notebook multiple times produces the same outputs.



## Code

In [6]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Constants representing the parameters of the model
ATTACK_RATE = 0.10
TRACE_SUCCESS = 0.20
SECONDARY_TRACE_THRESHOLD = 2

# Reproducibility control
SEED = 42

def simulate_event(m, rng):
  """
  Simulates the infection and tracing process for a series of events.
  Returns:
  - (p_wedding_infections, p_wedding_traces)
  """
  events = ['wedding'] * 200 + ['brunch'] * 800
  ppl = pd.DataFrame({
      'event': events,
      'infected': False,
      'traced': np.nan
  })

  ppl['traced'] = ppl['traced'].astype(pd.BooleanDtype())

  # Infect a random subset of people (simple random sample without replacement)
  infected_indices = rng.choice(ppl.index, size=int(len(ppl) * ATTACK_RATE), replace=False)
  ppl.loc[infected_indices, 'infected'] = True

  # Primary contact tracing (Bernoulli trials among infected)
  ppl.loc[ppl['infected'], 'traced'] = rng.random(sum(ppl['infected'])) < TRACE_SUCCESS

  # Secondary contact tracing (threshold-based amplification by event type)
  event_trace_counts = ppl[ppl['traced'] == True]['event'].value_counts()
  events_traced = event_trace_counts[event_trace_counts >= SECONDARY_TRACE_THRESHOLD].index
  ppl.loc[ppl['event'].isin(events_traced) & ppl['infected'], 'traced'] = True

  ppl['event_type'] = ppl['event'].str[0]
  wedding_infections = sum(ppl['infected'] & (ppl['event_type'] == 'w'))
  brunch_infections = sum(ppl['infected'] & (ppl['event_type'] == 'b'))
  p_wedding_infections = wedding_infections / (wedding_infections + brunch_infections)

  wedding_traces = sum(ppl['infected'] & ppl['traced'] & (ppl['event_type'] == 'w'))
  brunch_traces = sum(ppl['infected'] & ppl['traced'] & (ppl['event_type'] == 'b'))
  p_wedding_traces = wedding_traces / (wedding_traces + brunch_traces)

  return p_wedding_infections, p_wedding_traces

# Choose repetitions (try 10, 100, 1000)
REPS = 1000

rng = np.random.default_rng(SEED)
results = [simulate_event(m, rng) for m in range(REPS)]
props_df = pd.DataFrame(results, columns=["Infections", "Traces"])

plt.figure(figsize=(10, 6))
sns.histplot(props_df['Infections'], color="blue", alpha=0.75, binwidth=0.05, kde=False, label='Infections from Weddings')
sns.histplot(props_df['Traces'], color="red", alpha=0.75, binwidth=0.05, kde=False, label='Traced to Weddings')
plt.xlabel("Proportion of cases")
plt.ylabel("Frequency")
plt.title("Impact of Contact Tracing on Perceived Flu Infection Sources")
plt.legend()
plt.tight_layout()
plt.show()


ModuleNotFoundError: No module named 'pandas'

## Criteria

|Criteria|Complete|Incomplete|
|--------|----|----|
|Alteration of the code|The code changes made, made it reproducible.|The code is still not reproducible.|
|Description of changes|The author answered questions and explained the reasonings for the changes made well.|The author did not answer questions or explain the reasonings for the changes made well.|

## Submission Information
🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `23:59 - 06 January 2026`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This markdown file (`a1_sampling_and_reproducibility.ipynb`) should be populated with the code changed.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/sampling/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

#### Checklist:
- [ ] Create a branch called `assignment-1`.
- [ ] Ensure that the repository is public.
- [ ] Review [the PR description guidelines](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md#guidelines-for-pull-request-descriptions) and adhere to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via the help channel in Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
